# 02 — Spatially varying cluster plasma

**Question:** How well does a single-temperature regional analysis recover known temperature and abundance structure in an X-IFU simulation of Abell 2146?

**Success criterion:** distinguish continuous-map, discretized, detected, and fitted quantities; reconcile events with PHA counts; and quantify recovery bias.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from newathena_sixte_extended_sources import load_workspace

WORKSPACE = load_workspace()
ROOT = WORKSPACE.root
EXPOSURE = int(WORKSPACE.profile_values['phase2_exposure_s'])
RUNTIME = WORKSPACE.runtime / 'phase2-baseline'
INPUT = WORKSPACE.inputs / 'X-IFU_clusters_tutorial'
required = [
    RUNTIME / 'clusterA2146_arf_padded.simput',
    RUNTIME / f'multispec_{EXPOSURE}s_evt.fits',
    RUNTIME / f'multispec_{EXPOSURE}s.img',
    RUNTIME / f'phase2_{EXPOSURE}s_truth_summary.json',
    RUNTIME / f'phase2_{EXPOSURE}s_region_source_mixing.csv',
    RUNTIME / f'phase2_{EXPOSURE}s_fit_results.json',
]
for path in required:
    WORKSPACE.require(path, f'Phase 2 {WORKSPACE.profile} product')
print({'profile': WORKSPACE.profile, 'exposure_s': EXPOSURE,
       'science_status': WORKSPACE.profile_values['science_status']})

## Source truth

The co-registered surface-brightness, temperature, and abundance maps are approximated by eight logarithmic temperature values and eight linear abundance values.

In [ ]:
map_files = {
    'Surface brightness': 'A2146_SXB_russel_coord_cal.fits',
    'Temperature (keV)': 'A2146_kt_russel_coord_cal.fits',
    'Abundance (solar)': 'A2146_ab_russel_coord_cal.fits'}
maps = {label: fits.getdata(INPUT / name).astype(float) for label, name in map_files.items()}
assert len({array.shape for array in maps.values()}) == 1
assert all(np.isfinite(array).all() for array in maps.values())
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for axis, (label, array) in zip(axes, maps.items(), strict=True):
    shown = np.log10(np.clip(array, 1e-12, None)) if label == 'Surface brightness' else array
    image = axis.imshow(shown, origin='lower', cmap='magma')
    axis.set_title(label)
    fig.colorbar(image, ax=axis, fraction=0.046)
plt.show()

## Observation product contract

The deterministic 100 ks baseline uses X-IFU 1.11.1, the 4 eV no-filter configuration, and seed 20260721.

In [ ]:
with fits.open(RUNTIME / f'multispec_{EXPOSURE}s_evt.fits') as hdus:
    events = hdus['EVENTS'].data
    gti = hdus['STDGTI'].data
event_image = fits.getdata(RUNTIME / f'multispec_{EXPOSURE}s.img').astype(float)
summary = {'events': len(events), 'components': len(np.unique(events['SRC_ID'])),
           'image_counts': int(event_image.sum()),
           'gti': [float(gti['START'][0]), float(gti['STOP'][0])]}
assert summary['events'] == summary['image_counts']
if WORKSPACE.profile == 'reference':
    assert summary['components'] == 48
else:
    assert 0 < summary['components'] <= 48
plt.figure(figsize=(6, 5))
plt.imshow(np.log10(event_image + 1), origin='lower', cmap='viridis')
plt.colorbar(label='log10(counts + 1)')
plt.title(f'Simulated {EXPOSURE / 1000:g} ks X-IFU image ({WORKSPACE.profile})')
plt.show()
summary

## Region truth and source mixing

Continuous truth is brightness-weighted from the maps. Quantized truth uses the SIMPUT grid. Detected truth is weighted by recorded photons.

In [ ]:
truth = json.loads((RUNTIME / f'phase2_{EXPOSURE}s_truth_summary.json').read_text())
mixing = pd.read_csv(RUNTIME / f'phase2_{EXPOSURE}s_region_source_mixing.csv')
truth_table = pd.DataFrame(truth['regions']).T[[
    'continuous_brightness_weighted_temperature_keV',
    'quantized_brightness_weighted_temperature_keV',
    'detected_count_weighted_temperature_keV',
    'continuous_brightness_weighted_abundance_solar',
    'detected_count_weighted_abundance_solar',
    'region_event_rows', 'zero_signal_events', 'pha_counts']]
assert all(truth_table['region_event_rows'] - truth_table['zero_signal_events'] == truth_table['pha_counts'])
truth_table

In [ ]:
dominant = (mixing.sort_values(['region', 'detected_fraction'], ascending=[True, False])
            .groupby('region').head(8))
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for axis, region in zip(axes, ['cen', 'out'], strict=True):
    subset = dominant[dominant['region'] == region]
    axis.bar(subset['source_name'], 100 * subset['detected_fraction'])
    axis.set_title(f'{region}: leading components')
    axis.set_ylabel('Detected fraction (%)')
    axis.tick_params(axis='x', rotation=60)
plt.show()

## Naive single-temperature recovery

Ungrouped PHA files were fit over 0.2–12 keV with C-statistics and `phabs*apec`. Absorption and redshift are frozen consistently with the source model.

In [ ]:
fit_results = pd.DataFrame(json.loads((RUNTIME / f'phase2_{EXPOSURE}s_fit_results.json').read_text()))
recovery = fit_results[[
    'region', 'temperature_continuous_truth', 'temperature_keV',
    'temperature_bias_vs_continuous_fraction', 'abundance_continuous_truth',
    'abundance_solar', 'abundance_bias_vs_continuous_fraction']].copy()
recovery['temperature_bias_percent'] = 100 * recovery.pop('temperature_bias_vs_continuous_fraction')
recovery['abundance_bias_percent'] = 100 * recovery.pop('abundance_bias_vs_continuous_fraction')
central = fit_results.set_index('region').loc['cen']
outer = fit_results.set_index('region').loc['out']
if WORKSPACE.profile == 'reference':
    assert central['temperature_bias_vs_continuous_fraction'] < -0.25
    assert abs(outer['abundance_bias_vs_continuous_fraction']) < 0.05
recovery

## Try it: statistical precision versus physical bias

**Exercise.** Compare the teaching and reference profiles. Does a shorter exposure mainly change statistical precision, or does it remove the single-temperature modeling bias?

<details><summary>Solution and interpretation</summary>

The shorter run broadens and randomizes recovery, but it does not repair the physical mismatch between a multi-temperature region and a one-temperature model. The reference profile is required for the documented bias thresholds. If profile-specific products are missing, run the Phase 2 simulation, truth analysis, and fit with the matching `EXPOSURE` value.
</details>

## Interpretation and limitations

The multiphase center fits to about 2.52 keV rather than its 3.63 keV brightness-weighted truth. The approximately 31% bias is model mismatch, not a lack of counts. The hotter outer region is closer to truth but remains temperature-biased. This baseline excludes background and does not yet apply mixed-response correction.